# MNIST specialist → generalist → continuous Pareto exploration

Run the cells **from top to bottom**. Ten ResNet18 specialists learn one digit
versus the rest. A ResNet50 generalist sees two digits **overlaid on the same
28×28 canvas**. The first generalist run reproduces `mnist_okt`'s thresholded
teacher-label baseline. The second uses the ten unthresholded specialist
probabilities as ten competing distillation losses, then applies Frank–Wolfe
and MINRES tangent exploration.

The original executed notebook, including its old output, is preserved as
`mnist_okt_before_pareto.ipynb`. This notebook has no dependency on
`pareto_exploration.py` or `pareto_minres.py`.

Reference: Ma, Du, and Matusik, [*Efficient Continuous Pareto Exploration in
Multi-Task Learning*](https://arxiv.org/pdf/2006.16434), Equations (3)–(6),
Algorithm 1, and Section 5.

## 0. Requirements and imports

Use a Python environment with `torch`, `torchvision`, `numpy`, and
`matplotlib`. MNIST downloads automatically on the first run. A CUDA GPU is
strongly recommended; training eleven ResNets and taking second derivatives
can take a long time. If packages are missing, install them in your notebook
environment before running all cells.

In [ ]:
import copy
import json
import random
from dataclasses import dataclass
from math import hypot, sqrt
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

## 1. Settings

Start with the values below for the full experiment. If you only want a quick
check, reduce `SPEC_TRAIN_SIZE`, `STACK_TRAIN_SAMPLES`, and the epoch counts.
The Pareto section is a **local approximation**, not a proof that every point
is globally Pareto optimal.

In [ ]:
SEED = 42
BATCH_SIZE = 128
SPEC_EPOCHS = 3
SPEC_TRAIN_SIZE = 20_000
GEN_EPOCHS = 7
STACK_TRAIN_SAMPLES = 30_000
STACK_VAL_SAMPLES = 2_000
STACK_TEST_SAMPLES = 5_000

SOFT_SEED_EPOCHS = 2
CORRECTOR_EPOCHS = 1
PARETO_BETA_PAIRS = [(7, 9), (3, 8)]
TANGENT_BATCH_SIZE = 16
MINRES_MAX_ITER = 10
MINRES_TOLERANCE = 1e-3
RELATIVE_TANGENT_STEP = 0.01
PREFERENCE_STRENGTH = 0.5
INTERPOLATION_FRACTIONS = [0.5]

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Device:', DEVICE)

## 2. MNIST and a separate validation split

Specialists and the generalist train on `base_train`. Model selection and
Pareto pruning use `base_val`; the final `test_full` remains untouched until
the end. Each pair dataset samples fixed combinations from its base split.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_full = datasets.MNIST('./data', train=False, download=True, transform=transform)
split_indices = np.random.RandomState(SEED).permutation(len(train_full))
base_train = Subset(train_full, split_indices[:55_000].tolist())
base_val = Subset(train_full, split_indices[55_000:].tolist())
print('Original train:', len(train_full), '| training base:', len(base_train),
      '| validation base:', len(base_val), '| final test:', len(test_full))

## 3. Models and binary specialist data

Specialist `d` sees a positive label for digit `d` and a negative label for
every other digit. The ResNets start with random weights and are adapted to
one-channel 28×28 MNIST input.

In [ ]:
def make_resnet18(num_outputs=2):
    # Specialist: ResNet18 with a 2-output head -> [not_d, d]
    model = models.resnet18(weights=None)   # weights=None -> start from random weights
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.fc = nn.Linear(model.fc.in_features, num_outputs)
    return model

def make_resnet50(num_outputs=10):
    # Generalist: ResNet50 with a 10-output head -> one output per digit 0..9
    model = models.resnet50(weights=None)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.fc = nn.Linear(model.fc.in_features, num_outputs)
    return model

In [ ]:
class BinaryMNIST(Dataset):
    # Wraps MNIST and changes the label to: 1 if label == digit else 0
    def __init__(self, base_dataset, digit):
        self.base = base_dataset
        self.digit = digit

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        return img, int(label == self.digit)


if SPEC_TRAIN_SIZE is None:
    spec_train_base = base_train
else:
    sample_indices = np.random.permutation(len(base_train))[:SPEC_TRAIN_SIZE]
    spec_train_base = Subset(base_train, sample_indices.tolist())

## 4. Train and validate the ten specialists

The positive class gets weight 9 because one digit is much rarer than all
other digits combined. Training each model uses its own optimizer.

In [ ]:
def train_specialist(digit):
    model = make_resnet18(num_outputs=2).to(DEVICE)

    train_loader = DataLoader(BinaryMNIST(spec_train_base, digit),
                              batch_size=BATCH_SIZE, shuffle=True)

    # class 0 = "not d" (weight 1), class 1 = "d" (weight 9 because it is ~9x rarer)
    class_weights = torch.tensor([1.0, 9.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(SPEC_EPOCHS):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()              # 1) clear old gradients
            outputs = model(images)            # 2) forward pass
            loss = criterion(outputs, labels)  # 3) how wrong are we?
            loss.backward()                    # 4) compute gradients
            optimizer.step()                   # 5) update weights

            total_loss += loss.item()
        print(f"  specialist {digit} | epoch {epoch+1}/{SPEC_EPOCHS} | loss {total_loss/len(train_loader):.4f}")

    model.eval()
    torch.save(model.state_dict(), ARTIFACT_DIR / f"specialist_{digit}.pt")
    return model

In [ ]:
specialists = []
for d in range(10):
    # print(f"=== Training specialist for digit {d} ===")
    specialists.append(train_specialist(d))

In [ ]:
@torch.no_grad()
def evaluate_specialist(model, digit, base_dataset):
    loader = DataLoader(BinaryMNIST(base_dataset, digit), batch_size=256)
    correct, total, pos_correct, pos_total = 0, 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()
        pos_mask = labels == 1
        pos_correct += (preds[pos_mask] == 1).sum().item()
        pos_total += pos_mask.sum().item()
    return correct / total, pos_correct / max(pos_total, 1)

In [ ]:
for digit, specialist in enumerate(specialists):
    accuracy, recall = evaluate_specialist(specialist, digit, base_val)
    print(f'Specialist {digit}: validation accuracy {accuracy:.3f}, recall {recall:.3f}')

In [ ]:
@torch.no_grad()
def specialists_predict(images):
    # images: (B, 1, 28, 28) -> returns (B, 10): probability that each digit is present
    probs = []
    for m in specialists:
        m.eval()
        p = F.softmax(m(images), dim=1)[:, 1]   # index 1 = "it is digit d"
        probs.append(p)
    return torch.stack(probs, dim=1)

In [ ]:
validation_single_loader = DataLoader(base_val, batch_size=256)
correct = 0
for images, labels in validation_single_loader:
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    correct += (specialists_predict(images).argmax(dim=1) == labels).sum().item()
print(f'Specialist team validation accuracy: {correct / len(base_val):.3f}')

## 5. Build overlaid two-digit images

`StackedMNIST` returns five items: the overlay, its true multi-hot label, the
two digit IDs, and the two original images. The generalist sees only the
overlay. Teachers see each original image separately; **do not split the
overlay into left and right halves**.

In [ ]:
class StackedMNIST(Dataset):
    def __init__(self, base_dataset, n_samples, seed):
        self.base = base_dataset
        rng = np.random.RandomState(seed)
        # Pick which two MNIST images to overlay for every sample (fixed once, so it is repeatable)
        self.left_idx  = rng.randint(0, len(base_dataset), n_samples)
        self.right_idx = rng.randint(0, len(base_dataset), n_samples)

    def __len__(self):
        return len(self.left_idx)

    def __getitem__(self, i):
        img_a, lab_a = self.base[int(self.left_idx[i])]
        img_b, lab_b = self.base[int(self.right_idx[i])]

        # Superimpose the two digits on the same canvas -> (1, 28, 28)
        # max keeps both strokes at full brightness where they cross
        stacked = torch.maximum(img_a, img_b)

        target = torch.zeros(10)                     # multi-hot label
        target[lab_a] = 1.0
        target[lab_b] = 1.0
        return stacked, target, torch.tensor([lab_a, lab_b]), img_a, img_b

In [ ]:
stack_train = StackedMNIST(base_train, STACK_TRAIN_SAMPLES, seed=1)
stack_val = StackedMNIST(base_val, STACK_VAL_SAMPLES, seed=2)
stack_test = StackedMNIST(test_full, STACK_TEST_SAMPLES, seed=3)
train_loader = DataLoader(stack_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(stack_val, batch_size=BATCH_SIZE)
test_loader = DataLoader(stack_test, batch_size=BATCH_SIZE)
print('Pairs:', len(stack_train), 'train,', len(stack_val), 'validation,', len(stack_test), 'test')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for i, ax in enumerate(axes):
    image, target, digits, _, _ = stack_train[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'{digits.tolist()} → {torch.where(target == 1)[0].tolist()}')
    ax.axis('off')
plt.show()

## 6. Specialist teaching signals

The ten specialists make probability vectors for both original digits. The
pair target is `max` of those vectors, one value per digit. Keeping those
values unthresholded defines the **soft-label** Pareto objectives later.

For one overlay containing originals `a` and `b`, specialist `d` outputs
`p_d(a)` and `p_d(b)`. We set `q_d = max(p_d(a), p_d(b))`. If its confidence is
0.73, the teaching signal stays 0.73; turning it into 1 would discard
information. The generalist emits ten **logits** `z_d`, and the ten losses are

`L_d(θ) = mean_batch BCEWithLogits(z_d(θ), q_d)`, for `d = 0,…,9`.

`θ` means the generalist's trainable parameters. Every `L_d` is a separate
objective. One model must satisfy all ten, so improving one digit can worsen
another through the shared ResNet features. True digit labels are used later
to judge recognition; they do not define these distillation objectives.

In [ ]:
@torch.no_grad()
def soft_teacher_targets(img_a, img_b):
    """Ten unthresholded specialist hints for the two original MNIST images.

    The generalist sees their overlaid image, but the specialists see each
    original 28x28 image. Keeping probabilities here is essential for the
    soft-label distillation objectives in the tangent experiment.
    """
    return torch.maximum(specialists_predict(img_a.to(DEVICE)),
                         specialists_predict(img_b.to(DEVICE)))

def per_digit_distillation_losses(logits, teacher_probs):
    """L_d = mean over the batch of BCE(logit_d, teacher_probability_d)."""
    if logits.shape != teacher_probs.shape or logits.ndim != 2 or logits.shape[1] != 10:
        raise ValueError("Expected matching [batch, 10] logits and teacher probabilities")
    return F.binary_cross_entropy_with_logits(
        logits, teacher_probs, reduction="none").mean(dim=0)

## 7. Reproduce the `mnist_okt` thresholded-teacher baseline

For this baseline only, probabilities at least 0.5 become label 1. The model
trains on teacher decisions rather than true pair labels. Unlike the old
notebook, the optimizer is created for the model passed to this function.

In [ ]:
def train_generalist(model, loader, epochs, target_mode, lr=1e-3, save_file=None):
    if target_mode not in {'binary', 'soft'}:
        raise ValueError('target_mode must be binary or soft')
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for images, _, _, img_a, img_b in loader:
            images = images.to(DEVICE)
            teacher = soft_teacher_targets(img_a, img_b)
            if target_mode == 'binary':
                teacher = (teacher >= 0.5).float()
            logits = model(images)
            losses = per_digit_distillation_losses(logits, teacher)
            loss = losses.mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item()
        print(f'{target_mode} epoch {epoch + 1}/{epochs}: teacher BCE {total / len(loader):.4f}')
    model.eval()
    if save_file is not None:
        torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()},
                   ARTIFACT_DIR / save_file)
    return model

In [ ]:
generalist = make_resnet50(num_outputs=10).to(DEVICE)
generalist = train_generalist(generalist, train_loader, GEN_EPOCHS,
                             target_mode='binary', save_file='generalist_binary_teacher.pt')

## 8. Compare decision rules on validation data

The fixed 0.5 rule is the main metric. The gap, relative-max, and normalized
relative rules reproduce the extra decision experiments from `mnist_okt`.
Choose a rule on validation data only; the Pareto objectives themselves use
probabilities before any decision rule.

In [ ]:
@torch.no_grad()
def prediction_flags(probabilities, rule='fixed', ratio=0.5):
    if rule == 'fixed':
        return (probabilities > 0.5).float()
    if rule == 'gap':
        sorted_p, indices = probabilities.sort(dim=1, descending=True)
        k = (sorted_p[:, :-1] - sorted_p[:, 1:]).argmax(dim=1) + 1
        rank = torch.arange(10, device=probabilities.device).unsqueeze(0)
        return torch.zeros_like(probabilities).scatter(1, indices,
                 (rank < k.unsqueeze(1)).float())
    if rule == 'relative':
        return (probabilities > ratio * probabilities.max(dim=1, keepdim=True).values).float()
    if rule == 'normalized_relative':
        minimum = probabilities.min(dim=1, keepdim=True).values
        maximum = probabilities.max(dim=1, keepdim=True).values
        normalized = (probabilities - minimum) / (maximum - minimum + 1e-8)
        return (normalized > ratio).float()
    raise ValueError(rule)


@torch.no_grad()
def evaluate_decisions(model, loader, rule='fixed', ratio=0.5):
    model.eval()
    exact = total = 0
    correct = torch.zeros(10)
    for images, targets, *_ in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        probabilities = torch.sigmoid(model(images))
        predicted = prediction_flags(probabilities, rule, ratio)
        exact += (predicted == targets).all(dim=1).sum().item()
        correct += (predicted == targets).float().sum(dim=0).cpu()
        total += targets.size(0)
    return {'exact_match': exact / total, 'per_digit_accuracy': (correct / total).tolist()}


for rule, ratio in [('fixed', 0.5), ('gap', 0.5),
                    ('relative', 0.5), ('normalized_relative', 0.1)]:
    metrics = evaluate_decisions(generalist, val_loader, rule, ratio)
    print(f'{rule:20} validation exact match: {metrics["exact_match"]:.3f}')

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset=stack_val, count=6):
    model.eval()
    fig, axes = plt.subplots(1, count, figsize=(2.4 * count, 3))
    for ax in axes:
        image, _, digits, _, _ = dataset[random.randrange(len(dataset))]
        probabilities = torch.sigmoid(model(image.unsqueeze(0).to(DEVICE)))[0].cpu()
        predicted = torch.where(probabilities > 0.5)[0].tolist()
        truth = sorted(set(digits.tolist()))
        ax.imshow(image.squeeze(), cmap='gray')
        ax.set_title(f'true {truth}\npred {predicted}',
                     color='green' if predicted == truth else 'red')
        ax.axis('off')
    plt.show()


show_predictions(generalist)

## 9. Create a seed for ten soft-label objectives

Each objective `L_d` is the batch mean BCE between the student's digit-`d`
logit and specialist probability `q_d`. This experiment fine-tunes a copy of
the baseline using those soft probabilities. The true labels are retained
for external evaluation, not used in these ten distillation losses.

In [ ]:
soft_seed = copy.deepcopy(generalist)
soft_seed = train_generalist(soft_seed, train_loader, SOFT_SEED_EPOCHS,
                             target_mode='soft', lr=1e-4,
                             save_file='generalist_soft_seed.pt')
print('Soft seed validation:', evaluate_decisions(soft_seed, val_loader))
generalist.cpu()  # its weights are already saved; free GPU space for MINRES
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## 10. Frank–Wolfe: find a balance among ten task gradients

At a fixed model, let `g_d = gradient_theta L_d`. Frank–Wolfe solves
`min_alpha ||sum_d alpha_d g_d||²` on the simplex (`alpha >= 0`, sum 1).
The `10×10` Gram matrix `G Gᵀ` is enough. In the tangent function below,
the paper's equation uses unnormalized parameter gradients and `lam=0`.

Think of each `g_d` as the direction in parameter space that would increase
digit `d`'s loss fastest. A training step in the opposite direction lowers
that loss locally. The weighted sum `c = Σ alpha_d g_d` is a compromise across
digits. Frank–Wolfe starts with equal weights. At each iteration it picks the
simplex corner that most decreases `||c||²`, then moves partway toward it.

The weights are constrained to be nonnegative and sum to one. If the smallest
possible `||c||` is near zero, the model is near a **first-order Pareto
stationary** point in the selected parameter subspace. This condition alone
does not guarantee global Pareto optimality. In code, `G` has shape
`[10, number_of_selected_parameters]`, while `G @ G.T` is just `10×10`.

In [ ]:
@torch.no_grad()
def frank_wolfe_simplex(M, pref=None, lam=0.0, iters=20, tol=1e-6):
    """
    Frank-Wolfe solver on the probability simplex.
        min_alpha  alpha^T M alpha  +  lam * ||alpha - pref||^2
    - lam = 0         -> plain MGDA (one single min-norm point)
    - lam > 0 + pref  -> preference-guided MGDA (sweeping pref traces the front)
    Only the KxK Gram matrix M = G G^T is needed, so this is very cheap.
    """
    K = M.shape[0]
    alpha = torch.full((K,), 1.0 / K, device=M.device)      # start from uniform weights
    if pref is None:
        pref, lam = alpha.clone(), 0.0                      # no preference -> no proximal term
    pref = pref.to(M.device).float()

    # Hessian of the objective (constant, because the objective is quadratic)
    H = 2 * M + 2 * lam * torch.eye(K, device=M.device)

    for _ in range(iters):
        grad = H @ alpha - 2 * lam * pref                   # gradient of the objective
        j = torch.argmin(grad)                              # linear minimization oracle: best vertex e_j
        d = -alpha.clone()
        d[j] += 1.0                                         # direction d = e_j - alpha
        gap = -(grad @ d)                                   # Frank-Wolfe duality gap
        if gap < tol:                                       # converged
            break
        # exact line search for a quadratic, clipped to [0, 1] to stay inside the simplex
        gamma = (gap / (d @ H @ d).clamp_min(1e-12)).clamp(0.0, 1.0)
        alpha = alpha + gamma * d                           # convex combination keeps alpha on simplex
    return alpha

def _selftest():
    """Sanity check: Frank-Wolfe must match the closed-form 2-task MGDA solution."""
    g1, g2 = torch.tensor([1.0, 0.2]), torch.tensor([0.1, 1.0])
    G = torch.stack([g1, g2])
    alpha = frank_wolfe_simplex(G @ G.T, iters=200, tol=1e-10)
    closed = ((g2 @ (g2 - g1)) / ((g1 - g2) @ (g1 - g2))).clamp(0, 1)   # alpha for g1
    assert abs(alpha[0].item() - closed.item()) < 1e-3, (alpha, closed)
    assert abs(alpha.sum().item() - 1) < 1e-5 and (alpha >= 0).all()
    print("FW self-test passed:", alpha.tolist())

_selftest()

## 11. Matrix-free MINRES

The weighted Hessian has one row per model parameter and is far too large
to construct. MINRES receives a function that calculates `H @ v` using
automatic differentiation. The solver below stores only a few parameter
vectors, handles symmetric indefinite systems, and reports its measured
relative residual.

Here the Hessian `H` describes how the weighted gradient `c` changes when
model parameters move a little. `H @ v` is a Hessian-vector product, computed
by differentiating a gradient-vector product. The full Hessian is never saved.
MINRES finds a vector `v` whose product `H @ v` approximates a requested
right-hand side `b`. A small relative residual
`||H @ v - b|| / ||b||` indicates that this linear solve worked on the chosen
batch. It does not measure model accuracy or Pareto optimality.

In [ ]:
@dataclass
class MinresResult:
    solution: object
    iterations: int
    relative_residual: float
    converged: bool

def _norm(vector):
    return sqrt(max(float(vector @ vector), 0.0))

def minres(matvec, b, max_iter=50, tolerance=1e-4):
    """Approximately solve A x = b for symmetric (possibly indefinite) A.

    The stopping test uses the MINRES residual estimate. The returned residual
    is measured directly with one final matrix-vector product, so callers can
    tell whether a truncated solve is accurate enough for their experiment.
    """
    if max_iter < 1 or tolerance <= 0:
        raise ValueError("max_iter must be positive and tolerance must be > 0")
    x = b * 0
    beta_initial = _norm(b)
    if beta_initial == 0:
        return MinresResult(x, 0, 0.0, True)

    # Lanczos three-term recurrence, followed by Givens rotations of its
    # tridiagonal least-squares system. Only a few vectors of b's size live.
    r_previous = b
    r_current = b
    y = b
    beta_previous = 0.0
    beta = beta_initial
    dbar = epsln = 0.0
    cs, sn = -1.0, 0.0
    phibar = beta_initial
    w_previous = b * 0
    w_current = b * 0
    iterations = 0

    for iteration in range(1, max_iter + 1):
        v = y / beta
        y = matvec(v)
        if iteration > 1:
            y = y - (beta / beta_previous) * r_previous
        alpha = float(v @ y)
        y = y - (alpha / beta) * r_current
        r_previous, r_current = r_current, y
        beta_previous, beta = beta, _norm(r_current)

        old_epsln = epsln
        delta = cs * dbar + sn * alpha
        gbar = sn * dbar - cs * alpha
        epsln = sn * beta
        dbar = -cs * beta
        gamma = hypot(gbar, beta)
        if gamma == 0:
            break
        cs, sn = gbar / gamma, beta / gamma
        phi = cs * phibar
        phibar = sn * phibar

        w_old = w_previous
        w_previous = w_current
        w_current = (v - old_epsln * w_old - delta * w_previous) / gamma
        x = x + phi * w_current
        iterations = iteration
        if abs(phibar) <= tolerance * beta_initial or beta == 0:
            break

    residual = _norm(b - matvec(x)) / beta_initial
    return MinresResult(x, iterations, residual, residual <= tolerance)

In [ ]:
matrix = torch.tensor([[2.0, 0.0], [0.0, -1.0]])
rhs = torch.tensor([2.0, 1.0])
solver_check = minres(lambda vector: matrix @ vector, rhs, max_iter=5, tolerance=1e-6)
assert solver_check.converged
assert torch.allclose(solver_check.solution, torch.tensor([1.0, -1.0]), atol=1e-5)
print('MINRES check:', solver_check.iterations, 'iterations, relative residual',
      solver_check.relative_residual)

## 12. Tangent predictor: paper Equation (6)

Frank–Wolfe produces `alpha` and correction `c = sum alpha_d g_d`.
For a chosen `beta`, MINRES solves
`(sum alpha_d Hessian(L_d)) v = sum beta_d (g_d - c)`.
The direction `v` is a **model-parameter vector**, not ten task weights.

For example, `beta = e7 - e9` asks how model parameters can move when we
increase emphasis on digit 7 relative to digit 9. Because the entries of
this `beta` sum to zero, its right-hand side simplifies to `g_7 - g_9`.
The Hessian solve converts that **change of task emphasis** into a **change
of model parameters**. This is a local tangent near the soft seed. We try
both signs of a short step along it and then train each candidate again.

For a first run, select the last shared ResNet block plus the final head.
This controls memory but is a **subspace approximation**. Selecting only
`fc` would largely separate the ten tasks and hide their shared conflict.
Use all trainable parameters only when resources permit.

In [ ]:
def minres_teacher_tangent(model, batch, beta, params=None, max_iter=50,
                           tolerance=1e-4, damping=0.0):
    """Approximate the paper's tangent at a trained generalist checkpoint.

    Eq. (3): use Frank-Wolfe to find alpha minimizing ||sum_d alpha_d g_d||.
    Eq. (6): solve (sum_d alpha_d Hessian(L_d)) v =
            sum_d beta_d (g_d - c), where c = sum_d alpha_d g_d.

    ``batch`` must be from mnist_okt's overlaid StackedMNIST and contain
    (overlaid_images, true_targets, digits, original_a, original_b). The ten
    objectives use SOFT specialist probabilities, not its >= 0.5 labels.
    ``params`` defaults to all trainable student parameters. Passing a subset
    is cheaper, but explores only that parameter subspace. For a genuine
    trade-off, the subset must include shared parameters, not just the ten
    independent rows of the final classifier.

    This function returns a direction, not a complete Pareto front. A model
    perturbed along it still needs the paper's ParetoOptimize correction and
    validation before it can be claimed as another Pareto solution.
    """
    if damping < 0:
        raise ValueError("damping must be nonnegative")
    params = list(model.parameters() if params is None else params)
    params = [p for p in params if p.requires_grad]
    if not params:
        raise ValueError("No trainable parameters were selected")
    device = params[0].device
    if any(p.device != device or p.dtype != params[0].dtype for p in params):
        raise ValueError("Selected parameters must share a device and dtype")

    beta = torch.as_tensor(beta, dtype=torch.float32, device="cpu")
    if beta.shape != (10,) or not torch.isfinite(beta).all():
        raise ValueError("beta must be a finite length-10 vector")
    images, _, _, img_a, img_b = batch
    was_training = model.training
    model.eval()  # freeze BatchNorm statistics while taking Hessian products
    try:
        soft = soft_teacher_targets(img_a, img_b).to(device)
        logits = model(images.to(device))
        losses = per_digit_distillation_losses(logits, soft)

        # One gradient per objective. Offload rows to CPU to avoid storing
        # 10 full ResNet50 gradient vectors on the GPU at once.
        sizes = [p.numel() for p in params]
        G = torch.empty((10, sum(sizes)), dtype=torch.float32, device="cpu")
        for task, loss in enumerate(losses):
            grads = torch.autograd.grad(loss, params, retain_graph=True,
                                        allow_unused=True)
            offset = 0
            for p, grad, size in zip(params, grads, sizes):
                if grad is None:
                    G[task, offset:offset + size] = 0
                else:
                    G[task, offset:offset + size] = grad.detach().reshape(-1).to("cpu")
                offset += size

        # A common scaling changes neither Eq. (3)'s minimizer nor alpha,
        # and prevents very small gradient magnitudes from stalling FW.
        gradient_norms = G.norm(dim=1)
        scale = gradient_norms.max().clamp_min(1e-12)
        gram = (G @ G.T) / scale.square()
        alpha = frank_wolfe_simplex(gram, lam=0.0, iters=200, tol=1e-8)
        correction = alpha @ G
        rhs_cpu = beta @ G - beta.sum() * correction
        stationarity_ratio = (correction.norm() /
                              gradient_norms.mean().clamp_min(1e-12)).item()
        if rhs_cpu.norm() <= 1e-12:
            raise ValueError("The chosen beta has no tangent signal for these tasks")

        weighted_loss = (alpha.to(device) * losses).sum()
        first_grads = torch.autograd.grad(weighted_loss, params,
                                          create_graph=True, retain_graph=True,
                                          allow_unused=True)

        def hessian_vector_product(vector):
            parts = [chunk.view_as(p) for chunk, p in zip(vector.split(sizes), params)]
            active = [(grad, part) for grad, part in zip(first_grads, parts)
                      if grad is not None and grad.requires_grad]
            if active:
                second = torch.autograd.grad(
                    [item[0] for item in active], params,
                    grad_outputs=[item[1] for item in active],
                    retain_graph=True, allow_unused=True)
            else:
                second = [None] * len(params)
            flat = torch.cat([
                (grad.detach() if grad is not None else torch.zeros_like(p)).reshape(-1)
                for p, grad in zip(params, second)
            ])
            return flat + damping * vector

        rhs = rhs_cpu.to(device=device, dtype=params[0].dtype)
        solution = minres(hessian_vector_product, rhs, max_iter=max_iter,
                          tolerance=tolerance)
        length = solution.solution.norm()
        if not torch.isfinite(length) or length <= 1e-12:
            raise ValueError("MINRES returned a zero or non-finite direction")
        return {
            "direction": (solution.solution / length).detach(),
            "alpha": alpha.detach(),
            "beta": beta,
            "teacher_losses": losses.detach().cpu(),
            "stationarity_ratio": stationarity_ratio,
            "minres_iterations": solution.iterations,
            "minres_relative_residual": solution.relative_residual,
            "minres_converged": solution.converged,
            "parameter_count": sum(sizes),
        }
    finally:
        model.train(was_training)

@torch.no_grad()
def apply_tangent_step(params, direction, step_size):
    """Modify a COPY of a model's selected parameters by step_size * direction."""
    params = list(params)
    if direction.numel() != sum(p.numel() for p in params):
        raise ValueError("Direction length does not match the selected parameters")
    offset = 0
    for p in params:
        size = p.numel()
        p.add_(step_size * direction[offset:offset + size].view_as(p).to(p.device))
        offset += size

In [ ]:
def selected_parameters(model):
    return list(model.layer4[-1].parameters()) + list(model.fc.parameters())


tangent_loader = DataLoader(stack_train, batch_size=TANGENT_BATCH_SIZE,
                            shuffle=False)
calibration_batch = next(iter(tangent_loader))
tangents = []
for favored, traded in PARETO_BETA_PAIRS:
    beta = torch.zeros(10)
    beta[favored] = 1.0
    beta[traded] = -1.0
    result = minres_teacher_tangent(
        soft_seed, calibration_batch, beta,
        params=selected_parameters(soft_seed),
        max_iter=MINRES_MAX_ITER, tolerance=MINRES_TOLERANCE,
    )
    result['direction'] = result['direction'].cpu()
    tangents.append((favored, traded, result))
    print(f'beta=e{favored}-e{traded} | alpha={result["alpha"].tolist()}')
    print('  stationarity ratio:', result['stationarity_ratio'],
          '| MINRES relative residual:', result['minres_relative_residual'],
          '| converged:', result['minres_converged'])
    if not result['minres_converged']:
        print('  WARNING: tangent solve is inaccurate; increase MINRES_MAX_ITER.')
    if result['stationarity_ratio'] > 0.1:
        print('  WARNING: soft seed is far from first-order stationarity; '
              'treat this tangent as exploratory and train the seed longer.')

## 13. Predictor–corrector and validation

The paper expands `theta ± s v`, then applies `ParetoOptimize` to move each
candidate back toward a stationary trade-off. Here the corrector performs
short **positive-weight soft-label optimization** from each candidate.
Its weights tilt toward or away from the chosen `beta`. A one-epoch corrector
is a practical demonstration; increase `CORRECTOR_EPOCHS` and check residuals
before claiming close Pareto stationarity.

Dominance is evaluated on the **ten validation teacher losses**. Exact match
against true labels is reported separately: a model can imitate specialists
well without being the best digit recognizer.

Candidate A *dominates* B if A has no higher teacher loss on any of the ten
digits and is strictly lower on at least one. The code keeps candidates that
are not dominated by another **sampled** candidate. With only a few
candidates, this is a sampled non-dominated set, not the entire mathematical
Pareto front. The corrector's preference weights are positive and sum to one;
they give each candidate a different local trade-off during soft-label
training.

In [ ]:
def cpu_state_dict(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def correct_candidate(model, preference, epochs=CORRECTOR_EPOCHS):
    weights = preference.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for images, _, _, img_a, img_b in train_loader:
            teacher = soft_teacher_targets(img_a, img_b)
            logits = model(images.to(DEVICE))
            losses = per_digit_distillation_losses(logits, teacher)
            loss = (weights * losses).sum()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
        print(f'  corrector epoch {epoch + 1}/{epochs}: {running / len(train_loader):.4f}')
    model.eval()
    return model


@torch.no_grad()
def evaluate_front_point(model, loader):
    model.eval()
    total = exact = 0
    teacher_loss_sum = torch.zeros(10, device=DEVICE)
    tp = torch.zeros(10, device=DEVICE)
    fp = torch.zeros(10, device=DEVICE)
    fn = torch.zeros(10, device=DEVICE)
    for images, targets, _, img_a, img_b in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        teacher = soft_teacher_targets(img_a, img_b)
        logits = model(images)
        per_sample = F.binary_cross_entropy_with_logits(logits, teacher, reduction='none')
        teacher_loss_sum += per_sample.sum(dim=0)
        predicted = (torch.sigmoid(logits) > 0.5).float()
        exact += (predicted == targets).all(dim=1).sum().item()
        tp += (predicted * targets).sum(dim=0)
        fp += (predicted * (1 - targets)).sum(dim=0)
        fn += ((1 - predicted) * targets).sum(dim=0)
        total += images.size(0)
    f1 = 2 * tp / (2 * tp + fp + fn).clamp_min(1)
    return {
        'teacher_losses': (teacher_loss_sum / total).cpu().tolist(),
        'exact_match': exact / total,
        'f1_per_digit': f1.cpu().tolist(),
        'f1_mean': f1.mean().item(),
    }

In [ ]:
def pareto_front(points, maximize=True):
    """
    Return indices of non-dominated points.
    points: list of equal-length lists/tensors of objective values.
    A point is dominated if another is >= everywhere and > somewhere (if maximize).
    """
    P = torch.tensor([list(map(float, p)) for p in points])
    if not maximize:
        P = -P
    keep = []
    for i in range(len(P)):
        dominated = any(
            bool((P[j] >= P[i]).all() and (P[j] > P[i]).any())
            for j in range(len(P)) if j != i
        )
        if not dominated:
            keep.append(i)
    return keep

In [ ]:
seed_path = ARTIFACT_DIR / 'generalist_soft_seed.pt'
torch.save(cpu_state_dict(soft_seed), seed_path)
seed_metrics = evaluate_front_point(soft_seed, val_loader)
candidate_results = [dict(name='soft_seed', checkpoint=str(seed_path),
                          beta=None, sign=0, step_size=0.0, **seed_metrics)]

selected_norm = sqrt(sum(p.detach().float().square().sum().item()
                         for p in selected_parameters(soft_seed)))
step_size = RELATIVE_TANGENT_STEP * selected_norm
print('Selected-parameter norm:', selected_norm, '| tangent step:', step_size)

# Keep only the seed checkpoint on CPU while correcting new GPU models.
soft_seed.cpu()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

for favored, traded, tangent in tangents:
    for sign in (1, -1):
        name = f'tangent_{favored}_{traded}_{"plus" if sign > 0 else "minus"}'
        candidate = make_resnet50(num_outputs=10).to(DEVICE)
        candidate.load_state_dict(torch.load(seed_path, map_location=DEVICE,
                                             weights_only=True))
        apply_tangent_step(selected_parameters(candidate),
                           tangent['direction'], sign * step_size)
        preference = torch.softmax(
            torch.log(tangent['alpha'].clamp_min(1e-4)) +
            sign * PREFERENCE_STRENGTH * tangent['beta'], dim=0)
        candidate = correct_candidate(candidate, preference)
        metrics = evaluate_front_point(candidate, val_loader)
        checkpoint = ARTIFACT_DIR / f'{name}.pt'
        torch.save(cpu_state_dict(candidate), checkpoint)
        candidate_results.append(dict(name=name, checkpoint=str(checkpoint),
                                      beta=[favored, traded], sign=sign,
                                      step_size=sign * step_size,
                                      minres_residual=tangent['minres_relative_residual'],
                                      **metrics))
        print(name, '| exact:', round(metrics['exact_match'], 4),
              '| mean teacher BCE:', round(np.mean(metrics['teacher_losses']), 4))
        del candidate
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

front_indices = pareto_front([row['teacher_losses'] for row in candidate_results],
                             maximize=False)
for index, row in enumerate(candidate_results):
    row['nondominated'] = index in front_indices
    print(row['name'], '| nondominated:', row['nondominated'])

## 14. Sample a local continuous approximation

Section 5 of the paper interpolates between a corrected parent and nearby
corrected children. `interpolate_state` accepts **any** fraction `t` in
`[0,1]`; this run samples only the configured fractions to control cost.
Interpolated models are evaluated and labeled as approximations, never
automatically treated as stationary solutions. BatchNorm's floating buffers
are interpolated too.

For parent parameters `θ0`, child parameters `θ1`, and any `t` in `[0,1]`,
interpolation gives `θ(t) = (1-t)θ0 + tθ1`. That is a continuous curve in
**parameter space**. Whether its points have useful trade-offs must be
checked by evaluating their ten losses. The two-axis plot shows only digits
7 and 9 for readability; dominance still checks all ten digits.

In [ ]:
def interpolate_state(parent_state, child_state, fraction):
    if not 0.0 <= fraction <= 1.0:
        raise ValueError('fraction must be in [0, 1]')
    result = {}
    for key, parent_value in parent_state.items():
        child_value = child_state[key]
        if parent_value.is_floating_point():
            result[key] = torch.lerp(parent_value, child_value, fraction)
        else:
            result[key] = parent_value.clone() if fraction < 0.5 else child_value.clone()
    return result


parent_state = torch.load(seed_path, map_location='cpu', weights_only=True)
interpolated_results = []
for row in candidate_results[1:]:
    if not row['nondominated']:
        continue
    child_state = torch.load(row['checkpoint'], map_location='cpu', weights_only=True)
    for fraction in INTERPOLATION_FRACTIONS:
        local_model = make_resnet50(num_outputs=10).to(DEVICE)
        local_model.load_state_dict(interpolate_state(parent_state, child_state, fraction))
        metrics = evaluate_front_point(local_model, val_loader)
        interpolated_results.append(dict(parent='soft_seed', child=row['name'],
                                         fraction=fraction, **metrics))
        print('interpolate', row['name'], 't=', fraction,
              '| exact=', round(metrics['exact_match'], 4))
        del local_model
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

all_points = candidate_results + interpolated_results
sampled_front_indices = pareto_front([row['teacher_losses'] for row in all_points],
                                     maximize=False)
for index, row in enumerate(all_points):
    row['sampled_nondominated'] = index in sampled_front_indices

fig, ax = plt.subplots(figsize=(6, 5))
for row in candidate_results:
    ax.scatter(row['teacher_losses'][7], row['teacher_losses'][9],
               marker='o', color='tab:blue' if row['nondominated'] else 'lightgray')
    ax.annotate(row['name'], (row['teacher_losses'][7], row['teacher_losses'][9]), fontsize=7)
for row in interpolated_results:
    ax.scatter(row['teacher_losses'][7], row['teacher_losses'][9],
               marker='x', color='tab:orange')
ax.set_xlabel('Validation teacher BCE for digit 7 (lower is better)')
ax.set_ylabel('Validation teacher BCE for digit 9 (lower is better)')
ax.set_title('Two-objective view of a ten-objective local approximation')
plt.show()

## 15. Select on validation data, then test once

Choose among corrected, non-dominated checkpoints using validation exact
match against the true digit sets. The figure above shows only objectives 7
and 9; pruning used all ten teacher losses. The final test set is used here,
after the choice, for the first time.

In [ ]:
eligible = [row for row in candidate_results if row['nondominated']]
chosen = max(eligible, key=lambda row: row['exact_match'])
final_model = make_resnet50(num_outputs=10).to(DEVICE)
final_model.load_state_dict(torch.load(chosen['checkpoint'], map_location=DEVICE,
                                       weights_only=True))
final_metrics = evaluate_front_point(final_model, test_loader)
print('Selected by validation exact match:', chosen['name'])
print('Final test exact match:', final_metrics['exact_match'])
print('Final test mean per-digit F1:', final_metrics['f1_mean'])

summary = {
    'settings': {
        'seed': SEED, 'soft_seed_epochs': SOFT_SEED_EPOCHS,
        'corrector_epochs': CORRECTOR_EPOCHS,
        'minres_max_iter': MINRES_MAX_ITER,
        'relative_tangent_step': RELATIVE_TANGENT_STEP,
        'parameter_subspace': 'last ResNet50 layer4 block plus fc',
    },
    'corrected_candidates': candidate_results,
    'interpolated_samples': interpolated_results,
    'selected_checkpoint': chosen['checkpoint'],
    'final_test': final_metrics,
}
with (ARTIFACT_DIR / 'pareto_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)
print('Saved:', ARTIFACT_DIR / 'pareto_summary.json')